# Study report: irq-entry

The study measures the IRQ entry latency: the time from the edge to the
first instruction of the ISR. It uses the app `irq_latency` and four
interrupt paths. It also tests three possible causes of the modes and of
the tail: the wake-up of the CPU from WFI, flash or ART cache misses, and
time slicing.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. LATBENCH_INSTANCE: the instance folder ("study.py analysis notebooks" sets it). Else
# LATBENCH_RESULTS: the results root of a lab space (for example <lab>/results/kernel-latency), and the
# notebook finds its instance in it. LATBENCH_REPO: the safety repo; else the repo of the current folder.
import os
import subprocess
import sys
from pathlib import Path

INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or Path(
    os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-default-latencies@nucleo_g474re/st-irq-entry")
REPO = Path(os.environ.get("LATBENCH_REPO") or subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The modes of a histogram

A mode is a group of bins that each hold at least 0.1 % of the samples,
with at most 2 cycles between neighbours. Mode 1 is the fastest.
`mode_share(k)` of the checks is the share of the samples in mode k
(`hypotheses.yaml`, `scripts/benchmarks/studylib/stats.py`). The mode
tables use the same rule (`nbfig.modes`), so they show what the checks
see. This rule reads the fine bins only. Where a table has the columns
"... with coarse bins", these columns use the same rule over the fine
and the coarse bins: a coarse bin of 32 cycles is one bin. A difference
between the two views is a mode that the checks do not see.

The app data keep only the bins with samples. In the histograms, the
line goes down to a floor at each end of a group of bins, and an empty
bin is a gap.

## The four paths (irq-baseline-1m)

The study `zephyr-versions` owns this experiment. It has 1M samples per
point, the board defaults and an idle CPU. The paths (app page of
`irq_latency`):

- `stock`: the time stamp is in the user callback of the GPIO driver,
  after `_isr_wrapper` and the driver dispatch.
- `wrapped`: an ISR of the app, connected with `IRQ_CONNECT()`. It runs
  through `_isr_wrapper`.
- `direct`: a direct ISR of the app (`IRQ_DIRECT_CONNECT()`).
- `zli`: a direct zero-latency ISR. BASEPRI does not mask this IRQ, so
  kernel critical sections do not delay it.

The central result comes from the point `direct-idle`.

In [ ]:
if nbfig.built(inst, "irq-baseline-1m"):
    base = inst.rows("irq-baseline-1m")
    nbfig.dots(base, "IRQ entry per interrupt path (irq-baseline-1m)", stats=("p50", "p99", "p99.99", "max")).show()
    display(nbfig.pivot(base, ["p50", "p99", "p99.9", "p99.99", "max", "mean"]))

In [ ]:
if nbfig.built(inst, "irq-baseline-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "irq-baseline-1m"), "IRQ entry per interrupt path: CCDF (irq-baseline-1m)",
               ["irq"]).show()

## H1 and H2: the second mode

Two hypotheses give a cause of the second mode:

- H1: the wake-up of the CPU from WFI. With a busy CPU
  (`LATBENCH_BG_BUSY`), the idle thread does not run, and the CPU does
  not execute WFI. Thus `direct-busy` of `irq-smoke` has no second mode
  (H1 a).
- H2: flash or ART cache misses after other code ran. The main thread
  runs other code at each poll. Thus the share of the mode follows the
  poll rate (H2 a: `idle-poll1000` against `idle-poll10`). At 16 MHz the
  flash has 0 wait states, so a cache miss costs no time (H2 b:
  `idle-16mhz`).

The points of `irq-checks-1m` change one factor against `direct-idle` of
`irq-baseline-1m` (poll 100 ms, 170 MHz, idle CPU). `idle-85mhz` is a
clock between 170 MHz and 16 MHz. The column `check` gives the checks
that read a point, with the expectation from `hypotheses.yaml`.

In [ ]:
import plotly.graph_objects as go

EXPS = [e for e in ("irq-smoke", "irq-baseline-1m", "irq-checks-1m") if nbfig.built(inst, e)]
def expect_text(e):
    """The expectation of a check as a short text."""
    if "range" in e:
        return f"range {e['range'][0]}..{e['range'][1]}"
    return "increasing: " + " < ".join(e["order"]) if "order" in e else str(e)


CHECKS = {}
for hid, h in inst.design_hypotheses.items():
    for c in h.get("checks", []):
        if c.get("stat") == "mode_share(2)":
            pts = c.get("select", {}).get("point") or c["expect"].get("order", [])
            for p in [pts] if isinstance(pts, str) else pts:
                CHECKS.setdefault((c["experiment"], p), []).append(f"{hid} {c['id']}: {expect_text(c['expect'])}")

rows = []
for exp in EXPS:
    hz = inst.meta(exp).set_index("point").core_hz
    for run, tree, point, rep in inst.points(exp):
        m = nbfig.modes(rep["hists"]["irq"])
        rows.append({"experiment": exp, "point": point, "core MHz": hz[point] / 1e6, "modes": len(m),
                     "mode 1": (str(m[0][0]) if m[0][0] == m[0][1] else f"{m[0][0]}..{m[0][1]}") if m else "",
                     "share of mode 2": round(m[1][2], 4) if len(m) > 1 else 0.0,
                     "check": "; ".join(CHECKS.get((exp, point), []))})
share = pd.DataFrame(rows)
fig = go.Figure()
for k, exp in enumerate(EXPS):
    sub = share[share.experiment == exp]
    fig.add_trace(go.Bar(x=[sub.experiment, sub.point], y=sub["share of mode 2"], name=exp,
                         marker_color=nbfig.color(k), customdata=sub.check,
                         hovertemplate="%{x}: %{y:.4f}<br>%{customdata}<extra>" + exp + "</extra>"))
nbfig.style(fig, "Share of the second mode of the IRQ entry latency", "experiment / point",
            "mode_share(2)").show()
share.set_index(["experiment", "point"])

The histogram shows the modes near the p50. In `irq-checks-1m`, TIM2
runs at the core clock (both clock overlays keep APB1 at /1). Thus the
values of `idle-85mhz` and `idle-16mhz` are cycles of a slower clock.
The µs in the hover text use the clock of each point (LAT META
`core_hz`), and the column `core MHz` above gives it.

In [ ]:
series, clocks = {}, {}
for exp in EXPS:
    hists, hz = nbfig.hists_of(inst, exp), nbfig.clocks_of(inst, exp)
    for p in hists:
        if p in ("direct-idle", "direct-busy", "idle-poll10", "idle-poll1000", "idle-16mhz"):
            series[f"{exp} {p}"], clocks[f"{exp} {p}"] = hists[p], hz[p]
nbfig.hist(series, "IRQ entry near the modes: direct path", ["irq"], xrange=nbfig.mode_range(series, "irq", 8),
           hz=clocks).show()

## H3: time slicing and the tail with a busy CPU

The busy thread is preemptible, so it has a slice timeout. If time
slicing makes the larger tail, the tail of `busy-noslice`
(`TIMESLICING=n`) goes back to the idle level. The idle level is the
tail of `direct-idle` in `irq-baseline-1m` (same settings: poll 100 ms,
170 MHz, idle CPU). The check H3 a is manual: the design gives no tail
statistic and no tolerance for "the idle level".

In [ ]:
series, parts = {}, []
if nbfig.built(inst, "irq-baseline-1m"):
    series["irq-baseline-1m direct-idle"] = inst.hist("direct-idle", "irq", "irq-baseline-1m")
    parts.append(inst.rows("irq-baseline-1m").query("point == 'direct-idle'"))
for p in ("busy", "busy-noslice"):
    series[f"irq-checks-1m {p}"] = inst.hist(p, "irq", "irq-checks-1m")
parts.append(inst.rows("irq-checks-1m").query("point in ['busy', 'busy-noslice']"))
nbfig.ccdf({k: {"irq": h} for k, h in series.items()}, "IRQ entry tail: busy CPU with and without time slicing",
           ["irq"], dashed={"irq-baseline-1m direct-idle"}).show()
pd.concat(parts).set_index(["run", "point"])[["n", "p50", "p99", "p99.9", "p99.99", "max", "mean"]]